# Usage Data CSV Exporter (Tag-based)

Reads the `SystemLinkUsageTracking.UserRole.*` tag history from the Tag Historian and
writes a single long-format CSV for offline analysis. Run on demand (or as a
routine) and download the resulting file.

Each output row is one activity/role history point:

| column | meaning |
| --- | --- |
| `user_id` | SystemLink user id — raw GUID, or a keyed HMAC-SHA256 token when `User_Id_Output = "pseudonymized"` |
| `email_hash` *or* `email` | email column, controlled by `Email_Output` (see below); **omitted** when `Email_Output = "none"` |
| `activity_iso` | history value timestamp = latest activity time |
| `role` | user's role at that point: `operator`, `collaborator`, or empty (neither) |

**Output modes.** Two parameters control how the potentially-identifying columns
are written:

- **`Email_Output`** — how the user's email appears:
  - `"pseudonymized"` (default): an `email_hash` column holding a deterministic
    keyed HMAC-SHA256 token. This is the **only** mode whose files can be unioned
    and deduplicated across multiple SystemLink instances (see the Usage Data
    CSV Union notebook), because the same person yields the same token on every
    instance that shares the hardcoded secret.
  - `"plaintext"`: a raw `email` column. Convenient for a single trusted
    consumer, but writes real addresses into the file.
  - `"none"`: no email column at all. The exporter then also **skips the user
    email lookup**, so it never fetches any address — best for single-instance
    consumers that do not need cross-instance dedup.
- **`User_Id_Output`** — how the user id appears:
  - `"plaintext"` (default): the raw SystemLink user id (a re-identifiable GUID).
  - `"pseudonymized"`: a keyed HMAC-SHA256 token of the id instead, to reduce the
    personal data in the file. It remains a stable per-user pseudonym (all rows
    for one user share a token), just not the raw GUID.

**Choosing modes.** For cross-instance rollups keep `Email_Output =
"pseudonymized"`. For a single instance that does not need dedup, `Email_Output =
"none"` is the most private (no email is ever read). To further reduce PII, set
`User_Id_Output = "pseudonymized"`. Note that a CSV exported with `Email_Output`
of `"none"` or `"plaintext"` **cannot** be fed into the Usage Data CSV Union
notebook, which keys on `email_hash`; and even with both columns pseudonymized
the file is not truly anonymous (the tokens are stable per-person and, while
keyed by a shared secret that resists dictionary attacks, remain re-identifiable
by anyone holding that secret), so only share it with trusted consumers.

**Current-value fallback.** The Historian only keeps points inside the tag's
retention window and only when the tracker wrote a new point, so a user who has
not been active recently has no in-range history even though the tag still holds
a valid current value. When a tag has no history in the requested range, the
exporter falls back to the tag's **current value** (one row, using the current
value's timestamp) so those users still appear — giving a full-census export
rather than only recently-active users. A user is skipped only when it has
neither in-range history nor a current value inside the range.

**Email privacy.** When emails are pseudonymized, each address is normalized
(trimmed + lowercased) and turned into a keyed HMAC-SHA256 token (the full
64-hex digest) using the hardcoded `PSEUDONYMIZATION_SECRET`. Because the token
is deterministic and keyed by a shared secret, the same person yields the same
`email_hash` on every instance that uses the same secret, so exports can be
unioned and deduplicated across instances. Unlike an unkeyed hash, the token
cannot be reversed by a dictionary attack without also knowing the secret; keep
the secret confidential and only share the exported CSV with trusted consumers.


In [ ]:
import csv
import hashlib
import hmac
import os
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Set

import requests

# Parameters
Tag_Prefix = "SystemLinkUsageTracking"
Workspace = ""
Output_Path = "usage_tracking_export.csv"
Start_Time = ""  # ISO 8601; empty = from the beginning
End_Time = ""    # ISO 8601; empty = now

# How to represent the user's email in the CSV:
#   "pseudonymized" -> emit an `email_hash` column (deterministic keyed
#                      HMAC-SHA256 token). This is the ONLY mode whose files can be
#                      unioned/deduplicated across multiple SystemLink instances
#                      (see the Usage Data CSV Union notebook), because the same
#                      person yields the same token on every instance sharing the
#                      hardcoded secret.
#   "plaintext"     -> emit a raw `email` column. Convenient for a single
#                      trusted consumer, but writes real addresses to the file.
#   "none"          -> omit any email column. The exporter then also skips the
#                      user email lookup, so it never fetches any address at all
#                      (best for single-instance consumers that do not need it).
Email_Output = "pseudonymized"

# How to represent the user id in the CSV:
#   "plaintext"     -> emit the raw SystemLink user id (a re-identifiable GUID).
#   "pseudonymized" -> emit a keyed HMAC-SHA256 token of the user id instead, to
#                      reduce the personal data in the file. Note the id is still
#                      a stable per-user pseudonym (rows for one user share a
#                      token), it is just no longer the raw GUID.
User_Id_Output = "plaintext"

API_KEY = os.getenv("SYSTEMLINK_API_KEY")
BASE_URL = (os.getenv("SYSTEMLINK_HTTP_URI") or "").rstrip("/")
HEADERS = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "x-ni-api-key": API_KEY,
}

if not BASE_URL:
    raise ValueError("SYSTEMLINK_HTTP_URI environment variable must be set")

# Normalize and validate the output-mode parameters up front so a typo fails
# fast rather than silently producing an unexpected file.
Email_Output = (Email_Output or "").strip().lower()
if Email_Output not in ("none", "pseudonymized", "plaintext"):
    raise ValueError(
        'Email_Output must be one of "none", "pseudonymized", or "plaintext"; '
        f"got {Email_Output!r}."
    )

User_Id_Output = (User_Id_Output or "").strip().lower()
if User_Id_Output not in ("plaintext", "pseudonymized"):
    raise ValueError(
        'User_Id_Output must be one of "plaintext" or "pseudonymized"; '
        f"got {User_Id_Output!r}."
    )


# Resolve the Workspace parameter (which may be a workspace NAME, an ID, or
# blank) to a concrete workspace ID. The nitag/nitaghistorian queries filter by
# workspace ID, so passing a display name like "DTP" matches nothing. A blank
# value falls back to the caller's DEFAULT workspace.
def _request_with_retry(method, url, *, max_attempts=6, **kwargs):
    # Retry on rate limiting / transient server errors with exponential backoff,
    # honoring a Retry-After header when present. Enumerating tags and querying
    # per-user history in a loop can trip 429 Too Many Requests.
    retry_statuses = {429, 500, 502, 503, 504}
    backoff = 1.0
    last_resp = None
    for attempt in range(1, max_attempts + 1):
        resp = requests.request(method, url, **kwargs)
        if resp.status_code not in retry_statuses:
            return resp
        last_resp = resp
        if attempt == max_attempts:
            return resp
        retry_after = resp.headers.get("Retry-After")
        if retry_after:
            try:
                delay = float(retry_after)
            except ValueError:
                delay = backoff
        else:
            delay = backoff
        time.sleep(min(delay, 30.0))
        backoff = min(backoff * 2, 30.0)
    return last_resp


def _get_workspaces() -> List[Dict[str, Any]]:
    resp = _request_with_retry("GET", f"{BASE_URL}/niauth/v1/auth", headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json().get("workspaces", []) or []


def resolve_workspace_id(value: str) -> str:
    workspaces = _get_workspaces()
    name_to_id = {
        ws["name"].strip(): ws["id"].strip()
        for ws in workspaces
        if ws.get("name") and ws.get("id")
    }
    token = (value or "").strip()
    if not token:
        for ws in workspaces:
            if ws.get("default") and ws.get("id"):
                return ws["id"].strip()
        return ""
    if token in name_to_id:
        return name_to_id[token]
    if token in set(name_to_id.values()):
        return token  # already a workspace ID
    raise ValueError(
        f'Workspace "{token}" was not found. Available workspaces: '
        f'{", ".join(sorted(name_to_id)) or "(none)"}'
    )


Workspace = resolve_workspace_id(Workspace)
print(f"Using workspace ID: {Workspace!r}")

# Shared secret for the HMAC-SHA256 pseudonymization tokens. Hardcoded here on
# purpose so every instance produces the SAME token for a given person, which is
# what allows cross-instance union/dedup on `email_hash`. Unlike unkeyed SHA-256,
# an HMAC token cannot be reversed by a dictionary attack without also knowing
# this secret. IMPORTANT: replace the placeholder below with a private, random
# value before deploying, and use the exact SAME value on every instance whose
# exports are unioned together; changing it makes all previously exported tokens
# non-matching. Keep this value confidential.
_PLACEHOLDER_SECRET = "change-me-to-a-shared-random-secret"
PSEUDONYMIZATION_SECRET = "change-me-to-a-shared-random-secret"

# Fail fast when a pseudonymized column is requested but the secret is still the
# default placeholder. A publicly-known HMAC key would make the dictionary-attack
# protection illusory, so reject it rather than silently producing insecure
# tokens.
if "pseudonymized" in (Email_Output, User_Id_Output) and (
    not PSEUDONYMIZATION_SECRET or PSEUDONYMIZATION_SECRET == _PLACEHOLDER_SECRET
):
    raise ValueError(
        "PSEUDONYMIZATION_SECRET must be changed from its default placeholder to "
        "a private, non-default value when Email_Output or User_Id_Output is "
        '"pseudonymized". A missing or placeholder secret would make the HMAC '
        "key publicly known and defeat the dictionary-attack protection."
    )


def _hmac_token(text: str) -> str:
    """Return a deterministic, keyed HMAC-SHA256 token for a string.

    The input is normalized (trimmed + lowercased) before hashing so trivial
    formatting differences do not change the token. Returns "" for blank input.
    Uses the hardcoded PSEUDONYMIZATION_SECRET as the HMAC key, so the same
    person yields the same token on every instance that shares the secret, while
    remaining infeasible to reverse without it. Shared by the email and user-id
    pseudonymizers below.
    """
    norm = (text or "").strip().lower()
    if not norm:
        return ""
    return hmac.new(
        PSEUDONYMIZATION_SECRET.encode("utf-8"),
        norm.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()


def pseudonymize_email(email: str) -> str:
    """Return a deterministic, keyed HMAC token for an email.

    The address is normalized (trimmed + lowercased) and hashed with
    HMAC-SHA256 keyed by the hardcoded PSEUDONYMIZATION_SECRET, so the same
    person yields the same token on every instance that shares the secret, which
    is what allows a later cross-instance union/dedup on `email_hash`. Returns
    "" for a blank email (blank addresses cannot be deduplicated and are kept
    empty on purpose).

    The FULL 64-hex digest is emitted (previously truncated to 20 chars).
    Truncation shortened the token to 80 bits, which risks collisions where two
    different people share one `email_hash` and get merged during the union; the
    full digest removes that risk. Re-exports therefore produce a different (and
    longer) token than older files, so union only files exported at the same
    digest length AND with the same secret.

    Note: because the token is keyed by a shared secret, it is not reversible by
    a plain dictionary attack unless the attacker also knows the secret. Still
    only share the exported CSV with trusted consumers.
    """
    return _hmac_token(email)


def pseudonymize_user_id(user_id: str) -> str:
    """Return a deterministic, keyed HMAC-SHA256 token for a user id.

    Used when User_Id_Output == "pseudonymized" to keep a stable per-user key in
    the CSV without writing the raw, re-identifiable GUID. Like the email token
    it is keyed by the shared PSEUDONYMIZATION_SECRET, so it is a pseudonym
    rather than true anonymization.
    """
    return _hmac_token(user_id)


now_utc = datetime.now(timezone.utc)


def _iso_z(value: str, fallback: datetime) -> str:
    # The nitaghistorian query rejects timestamps ending in "+00:00"; it requires
    # a trailing "Z". Normalize whatever we send (parsed value or the default now)
    # to a Z-suffixed UTC string.
    token = (value or "").strip()
    if not token:
        dt = fallback
    else:
        parsed = token.replace("Z", "+00:00")
        try:
            dt = datetime.fromisoformat(parsed)
        except ValueError:
            return token  # leave non-ISO input untouched
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def _parse_iso_z(text: str) -> Optional[datetime]:
    """Parse a Z-suffixed (or offset) ISO timestamp to an aware UTC datetime.

    Returns None for blank/unparseable input. Used to range-check a tag's
    current value against the requested [Start_Time, End_Time] window.
    """
    token = (text or "").strip()
    if not token:
        return None
    try:
        dt = datetime.fromisoformat(token.replace("Z", "+00:00"))
    except ValueError:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


start_time = _iso_z(Start_Time, datetime(1970, 1, 1, tzinfo=timezone.utc))
end_time = _iso_z(End_Time, now_utc)
start_dt = _parse_iso_z(start_time)
end_dt = _parse_iso_z(end_time)


In [ ]:
def _enumerate_user_tags(prefix: str, workspace: str) -> List[str]:
    url = f"{BASE_URL}/nitag/v2/tags"
    paths: List[str] = []
    skip = 0
    take = 500
    pattern = f"{prefix}.UserRole.*"
    while True:
        params = {"path": pattern, "take": take, "skip": skip}
        if workspace:
            params["workspace"] = workspace
        resp = _request_with_retry("GET", url, headers=HEADERS, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        tags = payload.get("tags", []) if isinstance(payload, dict) else []
        for tag in tags:
            path = tag.get("path")
            if path:
                paths.append(path)
        total = payload.get("totalCount") if isinstance(payload, dict) else None
        skip += len(tags)
        if not tags or (total is not None and skip >= total):
            break
    return paths


def _read_tag_history(path: str, start: str, end: str, workspace: str) -> List[Dict[str, Any]]:
    url = f"{BASE_URL}/nitaghistorian/v2/tags/query-history"
    history: List[Dict[str, Any]] = []
    continuation_token = None
    while True:
        body = {"path": path, "startTime": start, "endTime": end, "take": 10000, "sortOrder": "ASCENDING"}
        if workspace:
            body["workspace"] = workspace
        if continuation_token:
            body["continuationToken"] = continuation_token
        resp = _request_with_retry("POST", url, headers=HEADERS, json=body, timeout=120)
        resp.raise_for_status()
        payload = resp.json()
        values = payload.get("values", []) if isinstance(payload, dict) else []
        history.extend(values)
        continuation_token = payload.get("continuationToken") if isinstance(payload, dict) else None
        if not continuation_token or not values:
            break
    return history


def _read_current_value(path: str, workspace: str) -> Optional[Dict[str, Any]]:
    """Return the tag's current value as {"timestamp", "value"} or None.

    Used as a fallback when the Tag Historian has no points for a tag in the
    requested range. The Historian only retains points for the retention window
    and only when the tracker actually wrote a new point, so a user who has not
    been active recently has a valid current value but no in-range history. The
    current value is the authoritative latest state, so falling back to it lets
    those users still appear in the export (full-census coverage).
    """
    url = f"{BASE_URL}/nitag/v2/tags/{workspace}/{path}/values"
    resp = _request_with_retry("GET", url, headers=HEADERS, timeout=30)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    # A 204 No Content / empty body means the tag exists but has never been
    # written, so there is no current value to fall back to.
    if resp.status_code == 204 or not (resp.content or b"").strip():
        return None
    current = (resp.json() or {}).get("current") or {}
    timestamp = current.get("timestamp")
    value = current.get("value")
    if isinstance(value, dict):
        value = value.get("value")
    if not timestamp:
        return None
    return {"timestamp": timestamp, "value": value}


def _in_range(timestamp: str) -> bool:
    """True when a current-value timestamp falls within [start_dt, end_dt]."""
    ts = _parse_iso_z(timestamp)
    if ts is None:
        return False
    if start_dt is not None and ts < start_dt:
        return False
    if end_dt is not None and ts > end_dt:
        return False
    return True


def _user_email_map() -> Dict[str, str]:
    url = f"{BASE_URL}/niuser/v1/users/query"
    mapping: Dict[str, str] = {}
    continuation_token = None
    while True:
        body = {"continuationToken": continuation_token, "take": 100}
        resp = _request_with_retry("POST", url, headers=HEADERS, json=body, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        for user in payload.get("users", []) if isinstance(payload, dict) else []:
            user_id = user.get("id")
            if user_id:
                mapping[user_id] = user.get("email", "")
        continuation_token = payload.get("continuationToken") if isinstance(payload, dict) else None
        if not continuation_token:
            break
    return mapping


# Per-user tag values are INT role codes written by the tracker. Decode them to
# the role name emitted in the CSV: 0 = neither, 1 = operator, 2 = collaborator.
CODE_TO_ROLE = {0: "", 1: "operator", 2: "collaborator"}


def _decode_role(value: Any) -> str:
    try:
        return CODE_TO_ROLE.get(int(str(value).strip()), "")
    except (TypeError, ValueError):
        return ""


tag_paths = _enumerate_user_tags(Tag_Prefix, Workspace)
# The email map is only needed when an email column is emitted. When
# Email_Output is "none" the niuser query is skipped entirely, so the exporter
# never fetches any email address.
need_email = Email_Output in ("pseudonymized", "plaintext")
email_map = _user_email_map() if need_email else {}
prefix_marker = f"{Tag_Prefix}.UserRole."

# Build the column list once so the header and every row stay in sync. The email
# column's header reflects the mode: "email_hash" for a pseudonymized token,
# "email" for a raw address, and no column at all for "none".
email_column_name = {"pseudonymized": "email_hash", "plaintext": "email"}.get(Email_Output)
columns = ["user_id"]
if email_column_name:
    columns.append(email_column_name)
columns += ["activity_iso", "role"]


def _email_field(user_id: str) -> Optional[str]:
    """Return the email cell for a user, or None when no email column is emitted."""
    if Email_Output == "pseudonymized":
        return pseudonymize_email(email_map.get(user_id, ""))
    if Email_Output == "plaintext":
        return email_map.get(user_id, "")
    return None


def _row(user_id_out: str, email_value: Optional[str], activity_iso: str, role: str) -> List[Any]:
    """Assemble one CSV row, including the email cell only when a column exists."""
    row: List[Any] = [user_id_out]
    if email_value is not None:
        row.append(email_value)
    row.append(activity_iso)
    row.append(role)
    return row


rows = 0
fallback_rows = 0
skipped_users = 0
duplicate_rows = 0
with open(Output_Path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(columns)
    for path in tag_paths:
        user_id = path[len(prefix_marker):] if path.startswith(prefix_marker) else path
        # Precompute the per-user output fields once (not per history point).
        user_id_out = pseudonymize_user_id(user_id) if User_Id_Output == "pseudonymized" else user_id
        email_value = _email_field(user_id)  # None when no email column is emitted
        history = _read_tag_history(path, start_time, end_time, Workspace)
        # Dedup within a user's history. The Tag Historian has no de-duplication
        # and can return byte-identical points (same timestamp + value) — e.g.
        # when a write is re-sent on a transient-error retry — so emit each
        # distinct (timestamp, role) at most once per user. This guarantees the
        # CSV is duplicate-free for consumers that do not dedup themselves.
        # (Keyed by (timestamp, value) rather than timestamp alone so a genuine
        # same-timestamp value change is still preserved.)
        seen: Set[tuple] = set()
        if history:
            for entry in history:
                timestamp = entry.get("timestamp", "")
                role = _decode_role(entry.get("value"))
                key = (timestamp, role)
                if key in seen:
                    duplicate_rows += 1
                    continue
                seen.add(key)
                writer.writerow(_row(user_id_out, email_value, timestamp, role))
                rows += 1
        else:
            # No in-range history: fall back to the tag's current value so the
            # user is still represented (unless the current value is itself
            # outside the requested range).
            current = _read_current_value(path, Workspace)
            if current and _in_range(current["timestamp"]):
                writer.writerow(
                    _row(user_id_out, email_value, current["timestamp"], _decode_role(current["value"]))
                )
                rows += 1
                fallback_rows += 1
            else:
                skipped_users += 1

print(
    f"Exported {rows} rows for {len(tag_paths)} users to {Output_Path} "
    f"(email_output={Email_Output!r}, user_id_output={User_Id_Output!r}; "
    f"{fallback_rows} from current-value fallback; "
    f"{duplicate_rows} duplicate points dropped; "
    f"{skipped_users} users had no in-range history or current value)"
)
